# PennyLane finite shots

Compare finite-shot Bell counts from default.qubit and MettleQ while keeping exact equality separate from statistical agreement.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [ ]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    total_variation_distance,
)

In [ ]:
shots = 4096
def make_counts(device):
    @qml.qnode(device)
    def circuit():
        qml.Hadamard(0)
        qml.CNOT(wires=[0, 1])
        return qml.counts(wires=[0, 1])
    return circuit

reference_device = qml.device("default.qubit", wires=2, shots=shots, seed=27)
reference_qnode = make_counts(reference_device)
reference, reference_ms, _ = benchmark(reference_qnode)
mettleq_device = MettleQDevice(wires=2, shots=shots, seed=27, method="statevector", device="cpu")
mettleq_qnode = make_counts(mettleq_device)
candidate, mettleq_ms, _ = benchmark(mettleq_qnode)
tvd = total_variation_distance(reference, candidate)
support_ok = set(reference) <= {"00", "11"} and set(candidate) <= {"00", "11"}
method, device = pennylane_selection(mettleq_device)
tutorial_result = emit_result(
    notebook="pennylane/02_finite_shots.ipynb",
    framework="pennylane",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="finite-shot total-variation distance <= 0.05",
    passed=support_ok and tvd <= 0.05,
    exact_match=reference == candidate,
    selected_method=method,
    selected_device=device,
    metrics={"tvd": tvd, "reference_counts": reference, "mettleq_counts": candidate},
    notes="Independent device RNG implementations need not return identical count dictionaries.",
)